# 12. Robotics Basic 심화 통합 강의

목표: ROS2, SLAM/Nav2, MoveIt2, OMX 모방학습, 비전, LLM/Ollama, MVP 통합을 `입력 -> 처리 -> 출력 -> 검증` 관점으로 이해한다.

이 노트북은 앞의 개별 노트북을 더 깊게 연결하는 중심 강의입니다. 용어를 외우는 대신, 데이터가 시스템 안에서 어떻게 흐르는지 따라가면서 학습합니다.

## 0. 전체 시스템을 한 문장으로 보기

로봇 시스템은 보통 다음 흐름으로 구성됩니다.

```text
센서 관측 -> 상태 추정 -> 지도/환경 이해 -> 계획 -> 제어 -> 실행 -> 로그/재계획
```

각 기술은 이 흐름의 특정 위치를 담당합니다.

| 기술 | 담당 위치 | 대표 입력 | 대표 출력 |
|---|---|---|---|
| ROS2 | 모듈 연결 | 메시지, 파라미터, launch | 실행 중인 노드 그래프 |
| TF2/URDF | 좌표계/로봇 모델 | 링크, 관절, transform | 좌표 변환 가능 상태 |
| SLAM | 위치추정+지도 | `/scan`, `/odom`, TF | `map`, `map->odom` |
| Nav2 | 자율주행 | 지도, 현재 pose, 목표 pose | `/cmd_vel`, 경로, 행동 상태 |
| MoveIt2 | 로봇팔 조작 | 목표 pose, planning scene | joint trajectory |
| 비전 | 물체/장면 인식 | image, camera info | box, mask, object pose |
| LLM/Ollama | 명령 해석 | 자연어 | 제한된 action sequence |
| 모방학습 | 행동 학습 | 관측+시연 action | policy action |

## 1. ROS2를 왜 먼저 배우는가

로봇은 하나의 거대한 프로그램이 아니라 여러 작은 프로그램의 집합입니다. 카메라 드라이버, LiDAR 드라이버, SLAM, Nav2, MoveIt2, LLM 명령 파서가 서로 데이터를 주고받아야 합니다. ROS2는 이 연결 구조를 표준화합니다.

- `Node`: 하나의 기능 단위입니다. 예: camera_node, slam_toolbox, planner_server
- `Topic`: 계속 흐르는 데이터입니다. 예: `/camera/image_raw`, `/scan`, `/cmd_vel`
- `Service`: 짧은 요청-응답입니다. 예: 지도 저장, 파라미터 변경
- `Action`: 오래 걸리는 목표 수행입니다. 예: 목적지까지 이동, 로봇팔 trajectory 실행
- `Launch`: 여러 노드와 파라미터를 한 번에 실행합니다.
- `URDF`: 로봇의 링크와 관절 구조를 설명합니다.
- `TF2`: 시간에 따라 변하는 좌표계 관계를 관리합니다.

### ROS2 사고 실험: 카메라 노드에서 YOLO 노드까지

```text
camera_node --/camera/image_raw--> yolo_node --/detections--> task_planner
```

여기서 카메라 노드는 YOLO가 있는지 몰라도 됩니다. YOLO 노드는 `/camera/image_raw`만 구독하면 됩니다. 이 분리 덕분에 USB 카메라를 Pi 카메라로 바꾸거나, YOLO를 다른 모델로 바꿔도 전체 시스템을 다시 쓰지 않아도 됩니다.

In [ ]:
# ROS2 topic 구조를 아주 작게 흉내낸 예제입니다.
# 실제 ROS2 코드는 아니지만 pub/sub 사고방식을 이해하는 데 도움이 됩니다.

class TopicBus:
    def __init__(self):
        self.subscribers = {}

    def subscribe(self, topic, callback):
        self.subscribers.setdefault(topic, []).append(callback)

    def publish(self, topic, message):
        for callback in self.subscribers.get(topic, []):
            callback(message)

bus = TopicBus()

def yolo_node(image):
    detections = [{"class": "cup", "bbox": [120, 80, 60, 90], "conf": 0.92}]
    bus.publish('/detections', detections)

def task_planner(detections):
    for det in detections:
        if det['class'] == 'cup' and det['conf'] > 0.8:
            print('Plan: pick cup candidate at bbox', det['bbox'])

bus.subscribe('/camera/image_raw', yolo_node)
bus.subscribe('/detections', task_planner)
bus.publish('/camera/image_raw', 'fake image frame')

## 2. TF2와 URDF가 중요한 이유

카메라가 본 물체 위치는 보통 `camera_frame` 기준입니다. 하지만 로봇팔은 `base_link` 기준 목표 pose가 필요하고, Nav2는 `map` 또는 `odom` 기준 목표 pose가 필요합니다.

예를 들어 카메라 좌표계에서 컵이 `(0.3m, 0.1m, 0.8m)`에 있다고 해도, 로봇팔 기준으로는 전혀 다른 값일 수 있습니다. TF2는 이런 좌표 변환을 일관되게 처리합니다.

```text
map -> odom -> base_link -> camera_link -> camera_optical_frame
                         -> arm_base -> wrist -> gripper
```

In [ ]:
# 2D 좌표 변환 예제: 센서 좌표의 점을 로봇 base 좌표로 변환합니다.
import math

def transform_2d(point, translation, yaw_deg):
    x, y = point
    tx, ty = translation
    yaw = math.radians(yaw_deg)
    c, s = math.cos(yaw), math.sin(yaw)
    return (c * x - s * y + tx, s * x + c * y + ty)

cup_in_camera = (0.4, 0.1)
camera_in_base_translation = (0.2, 0.0)
camera_in_base_yaw = 15

cup_in_base = transform_2d(cup_in_camera, camera_in_base_translation, camera_in_base_yaw)
print('cup in base_link:', cup_in_base)

## 3. SLAM의 핵심: 불확실한 위치와 지도를 함께 갱신하기

SLAM은 `내 위치를 알아야 지도를 만들 수 있고, 지도가 있어야 내 위치를 더 잘 알 수 있다`는 순환 문제입니다.

실무적으로는 다음 질문을 계속 확인합니다.

- `/scan`이 안정적으로 들어오는가?
- `/odom`이 너무 빠르게 drift하지 않는가?
- `base_link`, `laser`, `odom`, `map` TF가 끊기지 않는가?
- loop closure 후 지도가 찢어지거나 튀지 않는가?
- 지도 저장 후 Nav2 localization이 잘 되는가?

### Occupancy Grid 직관

2D SLAM 지도는 보통 격자 지도입니다. 각 칸은 비어 있음, 장애물, 미확인 상태를 가집니다.

- LiDAR ray가 지나간 칸: 비어 있을 가능성 증가
- LiDAR hit 지점: 장애물일 가능성 증가
- 관측하지 않은 칸: unknown 유지

아래 코드는 실제 SLAM은 아니지만, 관측으로 grid belief가 어떻게 바뀌는지 보여주는 toy 예제입니다.

In [ ]:
# Occupancy grid toy example
width, height = 12, 7
grid = [[0 for _ in range(width)] for _ in range(height)]  # 0 unknown, -1 free, 1 occupied

robot = (2, 3)
hit = (9, 3)

# 같은 행에서 로봇과 hit 사이를 free로 표시합니다.
for x in range(robot[0] + 1, hit[0]):
    grid[robot[1]][x] = -1
grid[hit[1]][hit[0]] = 1
grid[robot[1]][robot[0]] = 2

symbols = {0: '?', -1: '.', 1: '#', 2: 'R'}
for row in grid:
    print(' '.join(symbols[v] for v in row))

## 4. Nav2는 SLAM 이후의 문제를 푼다

SLAM이 지도를 만들었다면 Nav2는 그 지도 위에서 목표까지 이동합니다. Nav2의 핵심은 `경로를 계산하는 것`만이 아니라, 실패 상황을 감지하고 복구하는 전체 행동 흐름입니다.

| 구성 | 역할 | 실패 예시 |
|---|---|---|
| map server | 저장된 지도 제공 | 지도 파일 경로 오류 |
| localization | 지도 안에서 현재 위치 추정 | 초기 pose 부정확 |
| global planner | 큰 경로 계산 | 목표가 장애물 안에 있음 |
| local controller | 속도 명령 생성 | 좁은 통로에서 진동 |
| costmap | 장애물 비용 지도 | LiDAR frame/TF 오류 |
| behavior tree | 작업 흐름 관리 | recovery 반복 후 실패 |

입문자는 `경로가 안 나옴`을 planner 문제로만 보지 말고, TF, costmap, localization, footprint를 함께 봐야 합니다.

## 5. MoveIt2는 로봇팔의 '움직일 수 있는 경로'를 찾는다

MoveIt2의 문제는 Nav2와 다릅니다. Nav2는 평면에서 로봇 베이스를 움직이고, MoveIt2는 여러 관절을 가진 로봇팔이 충돌 없이 목표 pose에 도달하는 trajectory를 찾습니다.

Pick & Place를 한 번에 풀려고 하면 어렵습니다. 반드시 작업 단계를 나누어야 합니다.

```text
detect object -> compute grasp pose -> pre-grasp -> approach -> close gripper -> lift -> move -> lower -> open gripper
```

각 단계는 실패할 수 있습니다. 예를 들어 grasp pose가 벽 안에 있거나, pre-grasp에서 이미 충돌하거나, gripper가 물체를 놓칠 수 있습니다.

In [ ]:
# Pick & Place 작업 시퀀스를 상태 기계처럼 표현한 예제입니다.
steps = [
    {'name': 'detect_object', 'input': 'camera image', 'output': 'object pose'},
    {'name': 'plan_pregrasp', 'input': 'object pose', 'output': 'pregrasp joint trajectory'},
    {'name': 'approach', 'input': 'pregrasp pose', 'output': 'near object'},
    {'name': 'close_gripper', 'input': 'gripper command', 'output': 'object attached'},
    {'name': 'lift', 'input': 'attached object', 'output': 'safe height'},
    {'name': 'place', 'input': 'place pose', 'output': 'object released'},
]

for i, step in enumerate(steps, 1):
    print(f"{i}. {step['name']}: {step['input']} -> {step['output']}")

## 6. OMX 모방학습: 작은 VLA 실험으로 이해하기

OpenMANIPULATOR-X 같은 소형 로봇팔로 모방학습을 할 때 핵심은 모델이 아니라 데이터 구조입니다.

좋은 데이터 한 줄은 보통 이런 의미를 가집니다.

```json
{
  "image": "frame_000123.jpg",
  "joint_state": [0.1, -0.2, 0.3, 0.4],
  "gripper": 0.0,
  "instruction": "red block pick",
  "action": [0.02, 0.01, -0.03, 0.0, 1.0]
}
```

여기서 observation은 이미지와 현재 로봇 상태이고, action은 다음에 할 행동입니다. RT-1/RT-2 같은 큰 모델도 관점상 `관측 + 언어 -> 행동`을 학습한다는 점에서 같은 축에 있습니다.

In [ ]:
# 아주 작은 imitation policy 예제: 가장 비슷한 시연의 action을 따라 합니다.
# 실제 로봇에는 바로 쓰면 안 됩니다. 데이터 흐름 이해용입니다.

demos = [
    {'obs': [0.1, 0.1], 'action': 'move_left'},
    {'obs': [0.9, 0.1], 'action': 'move_right'},
    {'obs': [0.5, 0.8], 'action': 'close_gripper'},
]

def distance(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5

def nearest_demo_policy(obs):
    best = min(demos, key=lambda d: distance(d['obs'], obs))
    return best['action']

print(nearest_demo_policy([0.85, 0.2]))

## 7. OpenCV/YOLO와 LLM/Ollama는 역할이 다르다

비전 모델은 `무엇이 어디에 있는가`를 찾습니다. LLM은 `사용자가 뭘 하라고 했는가`를 구조화합니다. 둘을 섞어서 생각하면 위험합니다.

좋은 구조:

```text
자연어 -> LLM -> 제한된 action list
카메라 -> YOLO/OpenCV -> object 후보
object 후보 + TF -> 로봇 기준 pose
action list + pose -> Nav2/MoveIt2 실행
```

나쁜 구조:

```text
LLM이 자연어를 읽고 바로 모터 PWM 값을 출력
```

LLM 출력은 반드시 검증 가능한 중간 표현으로 제한해야 합니다.

In [ ]:
# LLM 출력으로 기대하는 형태의 예시입니다.
# 실제 Ollama 호출은 Basic/work/shared/04_ollama_command_parser.py를 참고하세요.

allowed_actions = {'detect_object', 'navigate_to', 'pick', 'place', 'stop'}
plan = [
    {'action': 'detect_object', 'target': 'red cup'},
    {'action': 'pick', 'target': 'red cup'},
    {'action': 'place', 'target': 'left box'},
]

def validate_plan(plan):
    for step in plan:
        if step['action'] not in allowed_actions:
            return False, f"not allowed: {step['action']}"
    return True, 'ok'

print(validate_plan(plan))

## 8. MVP를 만들 때의 최소 성공 기준

처음부터 완전한 자율 로봇을 만들려고 하면 실패 지점이 너무 많습니다. MVP는 다음 중 하나만 성공해도 충분합니다.

### 모바일 로봇 MVP

1. 카메라 또는 LiDAR topic이 ROS2에 들어온다.
2. SLAM 또는 localization이 현재 pose를 만든다.
3. Nav2가 목표 지점까지 이동한다.
4. 실패하면 로그와 bag으로 재현할 수 있다.

### 로봇팔 MVP

1. 카메라에서 물체 후보를 찾는다.
2. 물체 위치를 로봇 기준 좌표로 바꾼다.
3. MoveIt2가 pre-grasp pose까지 움직인다.
4. gripper open/close가 분리되어 테스트된다.

### 자연어 MVP

1. 사용자 명령을 제한된 action sequence로 바꾼다.
2. 각 action이 실제 ROS2 service/action 호출과 연결된다.
3. 실행 전 안전 검증이 있다.
4. 실패 시 멈추고 이유를 출력한다.

## 9. 다음 실습으로 연결

- Jetson/Raspberry Pi 실습: [13_edge_device_practical_labs.ipynb](13_edge_device_practical_labs.ipynb)
- 실제 실행 파일: [work/README.md](work/README.md)
- SLAM 실행 명령 생성: [work/slam_nav2/02_slam_nav2_command_builder.py](work/slam_nav2/02_slam_nav2_command_builder.py)
- 카메라/YOLO/Ollama 예제: [work/shared](work/shared)

학습 기준은 `설명 가능성`입니다. 각 모듈에 대해 입력, 출력, 실패 원인, 로그 확인 방법을 말할 수 있으면 다음 단계로 넘어갑니다.